# L5: 异步编程基础

> 理解 Python 异步编程，掌握 async/await

In [1]:
# === 环境设置 ===
import sys
import os
import time
from pathlib import Path

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

项目路径: c:\Users\zying\Desktop\pro_agent\agent-learning-project
Python 版本: 3.14.4


## 5.1 什么是异步编程？

**异步编程** 是一种并发编程方式，允许程序在等待某些操作（如网络请求）完成时，去执行其他任务。

### 为什么要异步？

```
同步（阻塞）:
  请求1 → 等待 → 响应1 → 请求2 → 等待 → 响应2
  总时间 = 等待1 + 等待2

异步（非阻塞）:
  请求1 + 请求2 → 等待（同时）→ 响应1 + 响应2
  总时间 = max(等待1, 等待2)
```

### 同步 vs 异步对比

In [2]:
import time
import asyncio

# 同步版本
def sync_task(name, seconds):
    print(f"{name}: 开始")
    time.sleep(seconds)  # 阻塞等待
    print(f"{name}: 完成")
    return f"{name}的结果"

def sync_main():
    start = time.time()
    
    result1 = sync_task("任务1", 1)
    result2 = sync_task("任务2", 1)
    result3 = sync_task("任务3", 1)
    
    elapsed = time.time() - start
    print(f"\n同步版本耗时: {elapsed:.2f}秒")
    print(f"结果: {[result1, result2, result3]}")

# 异步版本
async def async_task(name, seconds):
    print(f"{name}: 开始")
    await asyncio.sleep(seconds)  # 非阻塞等待
    print(f"{name}: 完成")
    return f"{name}的结果"

async def async_main():
    start = time.time()
    
    # 并发执行三个任务
    results = await asyncio.gather(
        async_task("任务1", 1),
        async_task("任务2", 1),
        async_task("任务3", 1),
    )
    
    elapsed = time.time() - start
    print(f"\n异步版本耗时: {elapsed:.2f}秒")
    print(f"结果: {results}")

# 运行对比
print("=== 同步版本 ===")
sync_main()

print("\n=== 异步版本 ===")
await async_main()

=== 同步版本 ===
任务1: 开始
任务1: 完成
任务2: 开始
任务2: 完成
任务3: 开始
任务3: 完成

同步版本耗时: 3.00秒
结果: ['任务1的结果', '任务2的结果', '任务3的结果']

=== 异步版本 ===
任务1: 开始
任务2: 开始
任务3: 开始
任务1: 完成
任务2: 完成
任务3: 完成

异步版本耗时: 1.00秒
结果: ['任务1的结果', '任务2的结果', '任务3的结果']


## 5.2 async/await 关键字

### 核心概念

| 关键字 | 作用 | 使用位置 |
|--------|------|----------|
| `async def` | 定义协程函数 | 函数定义 |
| `await` | 等待协程完成 | 函数内部 |

### 语法规则

In [3]:
import asyncio

# 定义一个简单的协程
async def say_hello(name):
    await asyncio.sleep(1)  # 模拟耗时操作
    return f"Hello, {name}!"

# 调用协程（错误方式）
try:
    result = say_hello("Alice")  # 返回协程对象，不会执行
    print(f"结果类型: {type(result)}")
    print(f"结果: {result}")
except:
    pass

# 正确方式：使用 await
result = await say_hello("Bob")
print(f"\n使用 await: {result}")

结果类型: <class 'coroutine'>
结果: <coroutine object say_hello at 0x00000272616ED8A0>

使用 await: Hello, Bob!


C:\Users\zying\AppData\Local\Temp\ipykernel_3060\145965430.py:17: RuntimeWarning: coroutine 'say_hello' was never awaited
  result = await say_hello("Bob")


## 5.3 asyncio 常用函数

| 函数 | 作用 |
|------|------|
| `asyncio.sleep()` | 异步睡眠，不阻塞事件循环 |
| `asyncio.gather()` | 并发执行多个协程 |
| `asyncio.create_task()` | 创建任务并调度执行 |
| `asyncio.wait()` | 等待一组任务完成 |
| `asyncio.run()` | 运行协程的入口 |

### 练习：并发获取多个数据

In [4]:
import asyncio

async def fetch_user(user_id):
    """模拟获取用户信息"""
    await asyncio.sleep(0.5)  # 模拟网络延迟
    return {"id": user_id, "name": f"User{user_id}"}

async def fetch_posts(user_id):
    """模拟获取用户文章"""
    await asyncio.sleep(0.3)
    return [
        {"id": 1, "title": f"Post by User{user_id}"},
        {"id": 2, "title": f"Another post by User{user_id}"},
    ]

async def fetch_comments(post_id):
    """模拟获取文章评论"""
    await asyncio.sleep(0.2)
    return [
        {"id": 1, "text": "Great post!"},
        {"id": 2, "text": "Thanks!"},
    ]

async def get_user_data(user_id):
    """获取用户完整数据（并发）"""
    start = asyncio.get_event_loop().time()
    
    # 并发获取用户信息和文章
    user, posts = await asyncio.gather(
        fetch_user(user_id),
        fetch_posts(user_id),
    )
    
    # 并发获取所有文章的评论
    comment_tasks = [fetch_comments(post["id"]) for post in posts]
    comments = await asyncio.gather(*comment_tasks)
    
    elapsed = asyncio.get_event_loop().time() - start
    
    return {
        "user": user,
        "posts": posts,
        "comments": comments,
        "time_elapsed": f"{elapsed:.2f}s",
    }

# 运行
result = await get_user_data(1)
print("用户数据:")
print(f"  用户: {result['user']}")
print(f"  文章数: {len(result['posts'])}")
print(f"  评论组数: {len(result['comments'])}")
print(f"  耗时: {result['time_elapsed']}")

用户数据:
  用户: {'id': 1, 'name': 'User1'}
  文章数: 2
  评论组数: 2
  耗时: 0.71s


## 5.4 并发 vs 并行

### 关键区别

```
并发 (Concurrency):
  - 同时处理多个任务
  - 同一时间段内交替执行
  - 适合 I/O 密集型任务
  - 示例：一边等网络响应，一边处理其他请求

并行 (Parallelism):
  - 同时执行多个任务
  - 同一时刻真正同时执行
  - 适合 CPU 密集型任务
  - 需要：多核 CPU、多进程
```

| 场景 | 推荐方式 |
|------|----------|
| 网络请求、数据库操作 | 异步 (asyncio) |
| 图像处理、机器学习 | 多进程 (multiprocessing) |
| 简单计算 | 多线程 (threading) |

## 5.5 项目中的异步实现

### 查看 LLM 客户端的异步方法

In [5]:
from backend.llm.base import BaseLLM
import inspect

print("=== BaseLLM 的异步方法 ===\n")

# 查看所有异步方法
for name, method in inspect.getmembers(BaseLLM, predicate=inspect.isfunction):
    if inspect.iscoroutinefunction(method):
        sig = inspect.signature(method)
        print(f"async {name}{sig}")

=== BaseLLM 的异步方法 ===

async chat(self, messages: List[backend.llm.models.Message], tools: List[backend.llm.models.ToolDefinition] | None = None, temperature: float = 0.7, max_tokens: int | None = None, **kwargs) -> backend.llm.models.ChatResponse
async embed(self, texts: List[str], **kwargs) -> backend.llm.models.EmbeddingResponse


### 为什么 LLM 调用必须是异步的？

```
1. 网络请求耗时：API 调用可能需要几秒
2. 并发处理：同时服务多个用户
3. 高效利用：等待响应时可以处理其他请求

示例：
  用户A 请求 → 发送API → 等待（同时处理用户B） → 返回结果
  用户B 请求 → 发送API → 等待（同时处理用户C） → 返回结果
```

## 5.6 异步上下文管理器

### 异步 with 语句

In [6]:
class AsyncConnection:
    """模拟异步连接管理器"""
    
    async def __aenter__(self):
        print("连接: 正在建立...")
        await asyncio.sleep(0.1)
        print("连接: 已建立")
        return self
    
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("连接: 正在关闭...")
        await asyncio.sleep(0.1)
        print("连接: 已关闭")
        return False
    
    async def query(self, sql):
        print(f"查询: {sql}")
        await asyncio.sleep(0.2)
        return f"{sql}的结果"

# 使用异步上下文管理器
async def use_connection():
    async with AsyncConnection() as conn:
        result = await conn.query("SELECT * FROM users")
        print(f"\n查询结果: {result}")
    # 退出时自动调用 __aexit__

await use_connection()

连接: 正在建立...
连接: 已建立
查询: SELECT * FROM users

查询结果: SELECT * FROM users的结果
连接: 正在关闭...
连接: 已关闭


## 5.7 异步迭代器

In [7]:
class AsyncStream:
    """模拟异步数据流"""
    
    def __init__(self, data):
        self.data = data
        self.index = 0
    
    def __aiter__(self):
        return self
    
    async def __anext__(self):
        await asyncio.sleep(0.1)  # 模拟数据到达延迟
        if self.index >= len(self.data):
            raise StopAsyncIteration
        value = self.data[self.index]
        self.index += 1
        return value

# 使用异步迭代器
async def process_stream():
    stream = AsyncStream(["数据1", "数据2", "数据3", "数据4"])
    
    print("异步迭代数据流:")
    async for item in stream:
        print(f"  处理: {item}")

await process_stream()

异步迭代数据流:
  处理: 数据1
  处理: 数据2
  处理: 数据3
  处理: 数据4


## 5.8 实战：模拟并发 LLM 调用

In [8]:
import asyncio
from pathlib import Path
from backend.llm import LLMFactory, Message

def get_deepseek_key():
    project_path = Path(os.getcwd())
    for path in [project_path] + list(project_path.parents):
        env_file = path / ".env"
        if env_file.exists():
            with open(env_file, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line.startswith("DEEPSEEK_API_KEY="):
                        return line.split("=", 1)[1].strip()
    return ""

async def ask_llm(question, index):
    """向 LLM 提问"""
    print(f"[任务{index}] 发送: {question}")
    
    api_key = get_deepseek_key()
    llm = LLMFactory.create("deepseek", api_key=api_key)
    
    start = asyncio.get_event_loop().time()
    response = await llm.chat([
        Message(role="user", content=question)
    ])
    elapsed = asyncio.get_event_loop().time() - start
    
    print(f"[任务{index}] 收到回答 (耗时{elapsed:.2f}s)")
    return {
        "question": question,
        "answer": response.content[:100] + "...",
        "time": elapsed,
    }

async def concurrent_questions():
    """并发提问多个问题"""
    questions = [
        "什么是Python？",
        "什么是异步编程？",
        "什么是LLM？",
    ]
    
    print("=== 并发提问 ===\n")
    start = asyncio.get_event_loop().time()
    
    # 并发执行
    results = await asyncio.gather(*[
        ask_llm(q, i+1) for i, q in enumerate(questions)
    ])
    
    total_time = asyncio.get_event_loop().time() - start
    
    print(f"\n=== 完成 ===")
    print(f"总耗时: {total_time:.2f}秒")
    print(f"平均耗时: {total_time/len(questions):.2f}秒/问题")
    
    return results

# 运行（取消注释以实际执行）
# await concurrent_questions()

## 5.9 常见面试问题

**Q: 异步和线程有什么区别？**
- A:
  | | 异步 (asyncio) | 线程 (threading) |
  |-|----------------|------------------|
  | **机制** | 事件循环，协作式多任务 | OS 线程，抢占式多任务 |
  | **开销** | 低（单线程）| 较高（创建线程）|
  | **适用** | I/O 密集型 | I/O 或 CPU 密集型 |
  | **GIL** | 不受 GIL 限制 | 受 GIL 限制 |
  | **复杂度** | 较高 | 较低 |

**Q: 什么是事件循环？**
- A: 事件循环是 asyncio 的核心，负责：
  1. 注册和执行协程
  2. 管理任务调度
  3. 处理 I/O 事件
  4. 管理定时器和回调
  ```python
  loop = asyncio.get_event_loop()  # 获取事件循环
  loop.run_until_complete(coro)     # 运行协程
  ```

**Q: await 后面可以接什么？**
- A: await 只能在 async 函数内使用，后面可以接：
  1. 协程函数：`await async_func()`
  2. 协程对象：`await coroutine`
  3. Task 对象：`await asyncio.create_task(coro)`
  4. Future 对象：`await future`

**Q: asyncio.gather() 和 asyncio.create_task() 有什么区别？**
- A:
  | | gather() | create_task() |
  |-|----------|---------------|
  | **返回** | 结果列表 | Task 对象 |
  | **等待** | 等待所有完成 | 调度后继续执行 |
  | **用途** | 需要所有结果时 | 不需要等待时 |

**Q: 如何取消正在运行的异步任务？**
- A: 使用 `task.cancel()` 方法：
  ```python
  task = asyncio.create_task(some_coro())
  await asyncio.sleep(0.1)
  task.cancel()  # 取消任务
  ```

**Q: 什么是协程？**
- A: 协程是一种用户态的轻量级线程：
  - 可以暂停和恢复执行
  - 由程序自己控制调度（不是 OS）
  - 适合协作式多任务
  - Python 中使用 `async def` 定义

**Q: 为什么 LLM 调用要用异步？**
- A:
  1. **I/O 等待**：网络请求耗时，等待时可以处理其他请求
  2. **高并发**：单线程可以服务多个用户
  3. **资源效率**：不需要为每个请求创建线程
  4. **框架要求**：FastAPI、Starlette 等 Web 框架原生支持异步

---

## 总结

本课程学习了 Python 异步编程的核心知识：

| 知识点 | 说明 |
|--------|------|
| **async/await** | 定义和调用协程 |
| **asyncio** | 异步 I/O 标准库 |
| **事件循环** | 协程调度器 |
| **并发 vs 并行** | 概念区别和选择 |
| **异步上下文管理器** | `async with` |
| **异步迭代器** | `async for` |

**记住**：异步编程适合 I/O 密集型任务（如网络请求、数据库操作），在 Agent 开发中至关重要！